# Análise de Resultados — LSTM CHiME6

Notebook para visualização e comparação de métricas do modelo treinado.
Carrega os artefatos salvos em `results/lstm_asr/` sem necessidade de rerodar o treino.

**Plots disponíveis:**
- AUROC comparativo por split (train / val / evaluation / test)
- Comparação entre métricas no conjunto de teste
- Matriz de confusão (absoluta e normalizada por linha)

In [ ]:
from pathlib import Path
import json
import ast

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

## 1. Carregar artefatos

In [ ]:
ARTIFACTS_DIR = Path('.') / 'results' / 'lstm_asr'

with open(ARTIFACTS_DIR / 'metrics_summary.json', 'r', encoding='utf-8') as f:
    payload = json.load(f)

df_metrics = pd.DataFrame(payload['metrics'])
print('Splits disponíveis:', df_metrics['split'].tolist())
display(df_metrics[['split', 'n_records', 'accuracy', 'precision', 'recall', 'f1', 'auroc']].round(4))

## 2. AUROC — comparativo por split

In [ ]:
_split_colors = {'train': '#4e79a7', 'val': '#59a14f', 'evaluation': '#f28e2b', 'test': '#e15759'}
_splits = df_metrics['split'].tolist()
_aurocs = df_metrics['auroc'].tolist()
_bar_colors = [_split_colors.get(s, '#aaa') for s in _splits]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(_splits, _aurocs, color=_bar_colors, width=0.5, edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, _aurocs):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.008,
            f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.9, label='Aleatório (0.5)')
ax.set_ylim(0, 1.12)
ax.set_ylabel('AUROC')
ax.set_title('AUROC por split')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'auroc_por_split.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Comparação de métricas — conjunto de teste

In [ ]:
_test_row = df_metrics[df_metrics['split'] == 'test'].iloc[0]
_metric_names = ['accuracy', 'precision', 'recall', 'f1', 'auroc']
_metric_vals  = [float(_test_row[m]) for m in _metric_names]
_metric_colors = ['#4e79a7', '#59a14f', '#f28e2b', '#e15759', '#b07aa1']

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(_metric_names, _metric_vals, color=_metric_colors, width=0.5,
              edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, _metric_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.008,
            f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.12)
ax.set_ylabel('Valor')
ax.set_title('Comparação de métricas — conjunto de teste')
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'metricas_teste.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Matriz de confusão — conjunto de teste

In [ ]:
_test_row = df_metrics[df_metrics['split'] == 'test'].iloc[0]
_cm = _test_row['confusion_matrix']
if isinstance(_cm, str):
    _cm = ast.literal_eval(_cm)
_cm_arr  = np.array([[_cm['tn'], _cm['fp']], [_cm['fn'], _cm['tp']]])
_cm_norm = _cm_arr / _cm_arr.sum(axis=1, keepdims=True)

_labels_x = ['Predito: ruído (0)', 'Predito: fala (1)']
_labels_y = ['Real: ruído (0)',    'Real: fala (1)']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, data, fmt, title in [
    (axes[0], _cm_arr,  'd',    'Contagens absolutas'),
    (axes[1], _cm_norm, '.2%', 'Recall por classe (normalizado por linha)'),
]:
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues', ax=ax,
                xticklabels=_labels_x, yticklabels=_labels_y,
                linewidths=0.5, linecolor='lightgray')
    ax.set_title(f'Matriz de confusão — teste\n({title})')
    ax.set_xlabel('Predito')
    ax.set_ylabel('Real')

plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'confusion_matrix_teste.png', dpi=150, bbox_inches='tight')
plt.show()